# extract important moments

In [ ]:
# %% [markdown]
# # Important moments from a single events CSV (notebook version)
# - Input: path to a StatsBomb events CSV for ONE match.
# - Output:
#     - `moments_df`: tidy DataFrame of important moments
#     - `moments_list`: JSON-safe list of dictionaries (ready for export)
# - Each item includes the ORIGINAL event `id` from the file.

import math
import json
from typing import Tuple, List, Dict, Any, Optional

import numpy as np
import pandas as pd
 

In [ ]:
def important_moments_from_file(
    filename: str,
    xg_threshold: float = 0.08,
) -> Tuple[pd.DataFrame, List[Dict[str, Any]]]:
    """
    Returns (moments_df, moments_list) for a match events CSV.
    Each moment includes the original event `id`.

    Moments:
      - Shots with xG >= xg_threshold OR shot_outcome == "Goal"
      - Own Goal Against
      - Foul Won with foul_won_penalty == True  (labeled as 'Penalty Won')

    Post-processing:
      - Filter out events after 120:00 if minute/second exist
      - Sort by period, minute, second, timestamp (when present)
      - If `possession` exists, keep the last event per possession (dedupe)
      - Make values JSON-safe (NaN/inf -> None; numpy scalars -> native)
    """
    df = pd.read_csv(filename)

    # --- Prepare shots & xG ---
    shots = df[df["type"] == "Shot"].copy() if "type" in df.columns else pd.DataFrame()
    if not shots.empty:
        if "shot_statsbomb_xg" in shots.columns:
            shots["xg"] = pd.to_numeric(shots["shot_statsbomb_xg"], errors="coerce")
        else:
            shots["xg"] = np.nan
        shot_mask = (shots["xg"] >= xg_threshold) | (shots.get("shot_outcome", "") == "Goal")
        shots = shots[shot_mask]
    # --- Own goals ---
    own_goals = df[df["type"] == "Own Goal Against"].copy() if "type" in df.columns else pd.DataFrame()
    if not own_goals.empty:
        own_goals["xg"] = np.nan  # no xG for own goals

    # --- Penalty won (foul won with penalty flag) ---
    pen_won = pd.DataFrame()
    if "type" in df.columns and "foul_won_penalty" in df.columns:
        pen_won = df[(df["type"] == "Foul Won") & (df["foul_won_penalty"] == True)].copy()
        if not pen_won.empty:
            pen_won["xg"] = np.nan  # not a shot, but keep xg column for schema consistency

    # --- Combine candidates ---
    moments = pd.concat([shots, own_goals, pen_won], ignore_index=True)

    # Nothing matched
    if moments.empty:
        return moments, []

    # --- Time filter: remove events strictly after 120:00 ---
    if {"minute", "second"}.issubset(moments.columns):
        mask = (moments["minute"] < 120) | ((moments["minute"] == 120) & (moments["second"].fillna(0) <= 0))
        moments = moments[mask].copy()

    # --- Sort by match order ---
    sort_cols = [c for c in ["period", "minute", "second", "timestamp"] if c in moments.columns]
    if sort_cols:
        moments = moments.sort_values(sort_cols, ascending=True, kind="stable")

    # --- Group by possession: keep last in each possession (if possession exists) ---
    if "possession" in moments.columns:
        # Secondary sort to ensure "last()" is deterministic with timestamp when available
        sort_cols2 = [c for c in ["possession", "period", "minute", "second", "timestamp"] if c in moments.columns]
        moments = moments.sort_values(sort_cols2, ascending=True, kind="stable")
        moments = moments.groupby("possession", as_index=False).last()

    # --- Drop exact duplicates if any ---
    moments = moments.drop_duplicates()

    # --- Create a tidy projection with stable columns ---
    def safe(val):
        if val is None:
            return None
        if isinstance(val, float) and (math.isnan(val) or math.isinf(val)):
            return None
        return val

    # Build a clean DataFrame with key fields; keep event `id`
    keep_cols = []
    for c in ["id","match_id","type","team","player","minute","second","timestamp","period","possession","xg","shot_outcome"]:
        if c in moments.columns:
            keep_cols.append(c)

    moments_df = moments[keep_cols].copy()

    # Label "Penalty Won" outcome for the Foul Won + flag case (nice-to-have)
    if {"type", "shot_outcome", "foul_won_penalty"}.issubset(moments.columns):
        mask_pw = (moments["type"] == "Foul Won") & (moments["foul_won_penalty"] == True)
        moments_df.loc[mask_pw.values, "shot_outcome"] = "Penalty Won"

    # JSON-safe list of dicts
    def json_fallback(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return None if np.isnan(o) else float(o)
        if isinstance(o, (np.bool_,)):
            return bool(o)
        return str(o)

    # Replace NaNs with None for JSON
    moments_df = moments_df.replace([np.nan, np.inf, -np.inf], None)

    moments_list: List[Dict[str, Any]] = []
    for _, row in moments_df.iterrows():
        # Compact "who" label like "C. Immobile"
        who = "N/A"
        player = row.get("player", None)
        if isinstance(player, str) and player.strip():
            parts = player.split()
            who = f"{parts[0][0]}. {parts[-1]}" if len(parts) >= 2 else parts[0]

        # "when" as mm:ss if minute/second present
        m, s = row.get("minute", None), row.get("second", None)
        when = f"{int(m)}:{int(s):02d}" if (m is not None and s is not None) else row.get("timestamp", None) or "N/A"

        item = {
            "event_id": row.get("id"),                           # <-- include original event id
            "match_id": row.get("match_id", None),
            "possession": row.get("possession", None),
            "who": who,
            "when": when,
            "event_type": row.get("type", None),
            "xg": row.get("xg", None),
            "outcome": row.get("shot_outcome", None),
            "team": row.get("team", None),
        }
        # JSON-safe coercion
        for k, v in list(item.items()):
            try:
                item[k] = v if v is None else json.loads(json.dumps(v, default=json_fallback))
            except Exception:
                item[k] = None if v is None else str(v)
        moments_list.append(item)

    return moments_df, moments_list


In [ ]:
FILENAME = "./events_by_match/events_3788741.csv"  # <-- put your file path here

moments_df, moments_list = important_moments_from_file(FILENAME, xg_threshold=0.08)

# Pretty print the DataFrame
with pd.option_context("display.max_columns", 50, "display.width", 180):
    print(moments_df.to_string(index=False))

# If you want to save:
# moments_df.to_csv("important_moments.csv", index=False)
# with open("important_moments.json", "w", encoding="utf-8") as f:
#     json.dump(moments_list, f, ensure_ascii=False, indent=2)


# classify shots

In [ ]:
# %% [markdown]
# # Classify "Shot" moments with multi-labels + xG
# - Inputs:
#     1) `filename` → path to the full events CSV for the match
#     2) `moments_df` → DataFrame returned by `important_moments_from_file(...)`
# - Output: `shot_moments_labeled` with columns:
#     ['id','match_id','team','player','minute','second','xg','distance_to_goal','labels','labels_str',
#      'shot_key_pass_id','shot_outcome']

import json, math
from typing import Optional, Tuple, List, Dict, Any
import numpy as np
import pandas as pd


In [ ]:
def parse_xy(val) -> Optional[Tuple[float, float]]:
    """Parse [x,y] stored as list/tuple or JSON-like string; None if not parseable."""
    if isinstance(val, (list, tuple)) and len(val) >= 2:
        try:
            return float(val[0]), float(val[1])
        except Exception:
            return None
    if isinstance(val, str):
        s = val.strip()
        if not s:
            return None
        try:
            j = json.loads(s.replace("'", '"'))
            if isinstance(j, (list, tuple)) and len(j) >= 2:
                return float(j[0]), float(j[1])
        except Exception:
            return None
    return None

def dist_to_nearest_goal(xy: Tuple[float, float]) -> float:
    """Euclidean distance from (x,y) to the nearest goal center in SB units (≈ yards)."""
    x, y = xy
    d_left  = math.hypot(x - 0.0,   y - 40.0)
    d_right = math.hypot(x - 120.0, y - 40.0)
    return min(d_left, d_right)

def get_pass_row_by_id(events: pd.DataFrame, pass_id: Any) -> Optional[pd.Series]:
    """Find the pass row by its `id` (string match after strip)."""
    if "id" not in events.columns:
        return None
    m = events[events["id"].astype(str).str.strip() == str(pass_id).strip()]
    return None if m.empty else m.iloc[0]


In [ ]:
# %% [markdown]
# ## Core: classify shots from the important moments list
def classify_shot_moments(
    filename: str,
    moments_df: pd.DataFrame,
    long_shot_yards: float = 22.0
) -> pd.DataFrame:
    """
    Classify important moments where type == 'Shot' with multi-labels and xG.

    Labels:
      - From key pass flags (on the pass row referenced by `shot_key_pass_id`):
          pass_through_ball, pass_cross, pass_cut_back
      - 'Long Shot' if distance to nearest goal center >= long_shot_yards
      - 'other' if none matched

    Returns a DataFrame with one row per shot moment.
    """
    # Load full events for lookups (shot_key_pass_id, locations, xg, etc.)
    ev = pd.read_csv(filename)

    # Tolerant type column
    type_col = "type" if "type" in ev.columns else ("Type" if "Type" in ev.columns else None)
    if type_col is None:
        raise ValueError("No 'type'/'Type' column in events CSV.")

    # Shots in the full events (we’ll use this to get fields not present in moments_df)
    ev_shots = ev[ev[type_col].astype(str).str.lower() == "shot"].copy()

    # Ensure an xg column exists on ev_shots
    if "shot_statsbomb_xg" in ev_shots.columns:
        ev_shots["xg"] = pd.to_numeric(ev_shots["shot_statsbomb_xg"], errors="coerce")
    else:
        ev_shots["xg"] = np.nan

    # Focus only on the events that are marked as important moments AND are shots
    if "type" not in moments_df.columns:
        raise ValueError("moments_df must include a 'type' column.")
    shot_ids = set(moments_df.loc[moments_df["type"] == "Shot", "id"]) if "id" in moments_df.columns else set()

    # If moments_df didn't carry 'id', fall back to intersect by time/team (not ideal). We require 'id'.
    if not shot_ids:
        raise ValueError("moments_df must include 'id' for shot events to classify them.")

    shots_focus = ev_shots[ev_shots["id"].isin(shot_ids)].copy()

    # Prepare output rows
    out_rows: List[Dict[str, Any]] = []

    for _, r in shots_focus.iterrows():
        labels: List[str] = []

        # 1) pass-based labels via key pass flags
        kp = r.get("shot_key_pass_id", np.nan)
        if pd.notna(kp):
            prow = get_pass_row_by_id(ev, kp)
            if prow is not None:
                # Only the three you asked for
                if "pass_through_ball" in prow.index and pd.notna(prow["pass_through_ball"]) and bool(prow["pass_through_ball"]):
                    labels.append("pass_through_ball")
                if "pass_cross" in prow.index and pd.notna(prow["pass_cross"]) and bool(prow["pass_cross"]):
                    labels.append("pass_cross")
                if "pass_cut_back" in prow.index and pd.notna(prow["pass_cut_back"]) and bool(prow["pass_cut_back"]):
                    labels.append("pass_cut_back")

        # 2) long shot (>= threshold yards to nearest goal center)
        loc = parse_xy(r.get("location"))
        dist_goal = float("nan")
        if loc is not None:
            dist_goal = dist_to_nearest_goal(loc)
            if dist_goal >= float(long_shot_yards):
                labels.append("Long Shot")

        # 3) other if nothing else matched
        if len(labels) == 0:
            labels.append("other")

        out_rows.append({
            "id": r.get("id"),
            "match_id": r.get("match_id"),
            "team": r.get("team"),
            "player": r.get("player"),
            "minute": r.get("minute"),
            "second": r.get("second"),
            "shot_outcome": r.get("shot_outcome"),
            "xg": r.get("xg"),
            "distance_to_goal": dist_goal,
            "labels": labels,
            "labels_str": ";".join(labels),
            "shot_key_pass_id": r.get("shot_key_pass_id"),
        })

    shot_moments_labeled = pd.DataFrame(out_rows)

    # Keep only the original important shot ids order if you want
    order_ids = moments_df.loc[moments_df["type"] == "Shot", "id"].tolist()
    if "id" in shot_moments_labeled.columns:
        shot_moments_labeled["__order"] = shot_moments_labeled["id"].apply(lambda x: order_ids.index(x) if x in order_ids else 10**9)
        shot_moments_labeled = shot_moments_labeled.sort_values("__order").drop(columns="__order")

    return shot_moments_labeled


In [ ]:
FILENAME = "./events_by_match/events_3788742.csv"  # <-- your match events CSV

# If you already have moments_df from earlier:
moments_df, moments_list = important_moments_from_file(FILENAME, xg_threshold=0.08)

shot_moments_labeled = classify_shot_moments(FILENAME, moments_df, long_shot_yards=22.0)

with pd.option_context("display.max_columns", 50, "display.width", 180):
    print(shot_moments_labeled.to_string(index=False))

# Optional save:
# shot_moments_labeled.to_csv("shot_moments_labeled_from_important.csv", index=False)


# classify own-goals and pens

In [ ]:
# %% [markdown]
# # Classify Own Goal / Penalty moments by nearest *prior* Pass (≤60s)
# Time handling:
# - Prefer absolute `minute` + `second`
# - Else use `period` + `timestamp` (timestamp is since start of period)
# Period offsets (SB): 1=0:00, 2=45:00, 3=90:00, 4=105:00, 5=penalties (ignored)

import json, math
from typing import Optional, Tuple, List, Dict, Any
import numpy as np
import pandas as pd


In [ ]:
# %% Helpers: time parsing and pass label extraction
PERIOD_OFFSETS = {1: 0, 2: 45*60, 3: 90*60, 4: 105*60}  # 5 = penalties, usually not timed like play

def parse_ts_seconds(ts: str) -> Optional[float]:
    """Parse 'MM:SS(.ms)' or 'HH:MM:SS(.ms)' → seconds."""
    if not isinstance(ts, str) or not ts.strip():
        return None
    s = ts.strip()
    try:
        parts = s.split(":")
        if len(parts) == 2:
            mm = int(parts[0]); ss = float(parts[1])
            return mm * 60.0 + ss
        elif len(parts) == 3:
            hh = int(parts[0]); mm = int(parts[1]); ss = float(parts[2])
            return hh * 3600.0 + mm * 60.0 + ss
    except Exception:
        return None
    return None

def event_time_seconds(row: pd.Series) -> Optional[float]:
    """
    Absolute match clock in seconds.
    Order:
      1) minute+second (already absolute across match in SB)
      2) period + timestamp (timestamp is per-period; add offset)
    """
    m, s = row.get("minute", None), row.get("second", None)
    if pd.notna(m) and pd.notna(s):
        try:
            return float(int(m) * 60 + int(s))
        except Exception:
            pass

    per = row.get("period", None)
    ts  = row.get("timestamp", None)
    ts_s = parse_ts_seconds(ts) if isinstance(ts, str) else None
    if pd.notna(per) and ts_s is not None:
        per = int(per)
        base = PERIOD_OFFSETS.get(per, 0)
        return base + ts_s

    return None

PASS_FLAG_COLUMNS = ["pass_through_ball", "pass_cross", "pass_cut_back"]

def pass_labels_from_row(pass_row: pd.Series) -> List[str]:
    labs: List[str] = []
    for c in PASS_FLAG_COLUMNS:
        if c in pass_row.index and pd.notna(pass_row[c]):
            try:
                if bool(pass_row[c]):
                    labs.append(c)
            except Exception:
                labs.append(c)
    return labs


In [ ]:
# %% Core: nearest prior pass (≤ window_sec) and classification
def classify_nonshot_moments_by_prior_pass(
    filename: str,
    moments_df: pd.DataFrame,
    window_sec: float = 60.0
) -> pd.DataFrame:
    """
    For important moments that are not shots (Own Goal Against, Penalty Won),
    find the closest *prior* Pass within `window_sec` and classify it by pass flags.

    Returns columns:
      ['moment_event_id','match_id','type','team','player','minute','second','timestamp','period',
       'nearest_pass_id','nearest_pass_time_s','delta_seconds','labels','labels_str']
    """
    ev = pd.read_csv(filename)

    # Precompute absolute seconds for all events
    ev["_t"] = ev.apply(event_time_seconds, axis=1)

    # Ensure IDs/types exist
    if "type" not in moments_df.columns or "id" not in moments_df.columns:
        raise ValueError("moments_df must include 'type' and 'id'.")

    # Ensure penalty flag visibility
    if "foul_won_penalty" not in moments_df.columns and "foul_won_penalty" in ev.columns:
        moments_df = moments_df.merge(ev[["id","foul_won_penalty"]], on="id", how="left")

    # Select target moments
    og_mask  = (moments_df["type"] == "Own Goal Against")
    pen_mask = (moments_df["type"] == "Foul Won") & (moments_df.get("foul_won_penalty", False) == True)
    focus = moments_df[og_mask | pen_mask].copy()
    if focus.empty:
        return pd.DataFrame(columns=[
            "moment_event_id","match_id","type","team","player","minute","second","timestamp","period",
            "nearest_pass_id","nearest_pass_time_s","delta_seconds","labels","labels_str"
        ])

    # Pass table with absolute times
    passes = ev[ev["type"] == "Pass"].copy() if "type" in ev.columns else pd.DataFrame()
    if not passes.empty and "_t" not in passes.columns:
        passes["_t"] = passes.apply(event_time_seconds, axis=1)
    passes = passes[passes["_t"].notna()].copy()

    rows: List[Dict[str, Any]] = []

    for _, m in focus.iterrows():
        t = event_time_seconds(m)
        nearest = None
        if t is not None and not passes.empty:
            # only prior passes within window_sec
            cand = passes.copy()
            cand["_delta"] = t - cand["_t"]
            cand = cand[(cand["_delta"] >= 0) & (cand["_delta"] <= float(window_sec))]
            if not cand.empty:
                nearest = cand.loc[cand["_delta"].idxmin()]

        labels = pass_labels_from_row(nearest) if nearest is not None else []
        rows.append({
            "moment_event_id": m.get("id"),
            "match_id": m.get("match_id"),
            "type": m.get("type"),
            "team": m.get("team"),
            "player": m.get("player"),
            "minute": m.get("minute"),
            "second": m.get("second"),
            "timestamp": m.get("timestamp"),
            "period": m.get("period"),
            "nearest_pass_id": (nearest.get("id") if nearest is not None else None),
            "nearest_pass_time_s": (nearest.get("_t") if nearest is not None else None),
            "delta_seconds": (float(nearest["_delta"]) if nearest is not None else None),
            "labels": labels,
            "labels_str": ";".join(labels) if labels else "",
        })

    out = pd.DataFrame(rows)

    # Sort by match order if available
    sort_cols = [c for c in ["period","minute","second","timestamp"] if c in out.columns]
    if sort_cols:
        out = out.sort_values(sort_cols, ascending=True, kind="stable")

    return out


In [ ]:
# %% Example usage
FILENAME = "./events_by_match/events_3788741.csv"  # <-- your match events CSV

moments_df, moments_list = important_moments_from_file(FILENAME, xg_threshold=0.08)

nonshot_prior_1min = classify_nonshot_moments_by_prior_pass(
    filename=FILENAME,
    moments_df=moments_df,
    window_sec=60.0  # 1 minute prior only
)

with pd.option_context("display.max_columns", 80, "display.width", 200):
    print(nonshot_prior_1min.to_string(index=False))

# Optional: save
# nonshot_prior_1min.to_csv("nonshot_moments_labeled_prior_60s.csv", index=False)
